# Enrichment

Enrichment adds message-level theme and sentiment labels to locally stored customer-message data. It reads Preparation artifacts and writes local enriched artifacts for the Analysis phase.

---

## Libraries

Imports the standard and tabular-data libraries required by the notebook. Model-specific dependencies are added here only after the local model is selected.

In [ ]:
import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == "enrichment":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from enrichment.modules.ollama import generate_json, list_models

---

## Import

Loads the prepared ticket dataset from `enrichment/data/` for enrichment. **Run Preparation through Export before running this section.**

In [ ]:
project_root = Path.cwd()
if project_root.name == "enrichment":
    project_root = project_root.parent

prepared_tickets_path = (
    project_root / "enrichment" / "data" / "prepared_tickets.parquet"
)
if not prepared_tickets_path.is_file():
    raise FileNotFoundError(
        "Run the Preparation notebook through Export before importing data."
    )

prepared_tickets = pd.read_parquet(prepared_tickets_path)

prepared_tickets

---

## Text Profiling

Validates the structure and coverage of customer-message sequences before classification. Message collections remain separate within each ticket so enrichment can preserve theme and sentiment changes across messages.

### Message Structure

Normalizes the Parquet-loaded message collections into Python lists, then verifies their contents and counts. The source `customer_message_count` field provides an independent check on each ticket's message sequence.

In [ ]:
message_collection_is_valid = prepared_tickets["customer_messages"].map(
    lambda value: isinstance(value, (list, np.ndarray))
)
if not message_collection_is_valid.all():
    invalid_count = (~message_collection_is_valid).sum()
    raise TypeError(
        f"customer_messages contains {invalid_count} non-list-like collections."
    )

message_lists = prepared_tickets["customer_messages"].map(
    lambda value: value.tolist() if isinstance(value, np.ndarray) else value
)
message_lengths = message_lists.map(len)
flat_messages = [message for messages in message_lists for message in messages]

message_structure_profile = pd.DataFrame(
    {
        "metric": [
            "tickets",
            "message_collection_type",
            "total_messages",
            "empty_message_list_count",
            "non_string_message_count",
            "empty_or_whitespace_message_count",
            "minimum_messages_per_ticket",
            "median_messages_per_ticket",
            "maximum_messages_per_ticket",
            "source_count_mismatch_count",
        ],
        "value": [
            len(prepared_tickets),
            prepared_tickets["customer_messages"].map(
                lambda value: type(value).__name__
            ).mode().iat[0],
            len(flat_messages),
            (message_lengths == 0).sum(),
            sum(not isinstance(message, str) for message in flat_messages),
            sum(
                isinstance(message, str) and not message.strip()
                for message in flat_messages
            ),
            message_lengths.min(),
            message_lengths.median(),
            message_lengths.max(),
            (message_lengths != prepared_tickets["customer_message_count"]).sum(),
        ],
    }
)

message_structure_profile

#### *Findings*

All 10,000 ticket message collections load as NumPy arrays, each containing only strings and matching the source `customer_message_count`. There are 25,234 messages, no empty message lists or blank messages, and sequences range from 2 to 12 messages. The source provides no message-level timestamp, so later enrichment preserves source-array order as message position without interpreting it as chronological order.

---

### Message Distribution

Counts customer messages per ticket to establish the sequence lengths used by later theme and sentiment comparisons. The distribution uses tickets as its denominator.

In [ ]:
message_count_distribution = (
    message_lengths.value_counts()
    .sort_index()
    .rename_axis("message_count")
    .reset_index(name="tickets")
    .assign(ticket_share=lambda frame: frame["tickets"] / len(prepared_tickets))
)

message_count_distribution

#### *Findings*

Two-message tickets account for 67.1% of the dataset, while 95.7% contain two to four customer messages. The remaining 4.3% form a small long tail of five to 12 messages, so sequence-level comparisons remain possible without allowing longer conversations to change ticket-level denominators.

---

## Taxonomy

Creates disjoint prompt-development and validation samples for defining and evaluating the text-derived theme taxonomy. Source-system contact labels balance selection coverage but do not determine the labels assigned to customer messages.

### Source Coverage

Selects two tickets from each available `main_contact_reason` value, using a fixed seed for reproducibility, then assigns one to each sample. **Each 15-ticket sample is judged sufficient to demonstrate prompt development and validation.** A production system would use larger samples.

In [ ]:
tickets_per_group = 2
taxonomy_random_seed = 42
sampling_labels = prepared_tickets["main_contact_reason"].fillna("Missing")
sampling_groups = list(prepared_tickets.groupby(sampling_labels, dropna=False))
sampled_ticket_indices = []

for _, group in sampling_groups:
    if len(group) < tickets_per_group:
        raise ValueError(
            "Each source-label group needs at least two tickets for the split."
        )
    sampled_ticket_indices.extend(
        group.sample(n=tickets_per_group, random_state=taxonomy_random_seed).index
    )

taxonomy_sampled_tickets = prepared_tickets.loc[sampled_ticket_indices].copy()
taxonomy_sampled_tickets["sampling_label"] = sampling_labels.loc[
    taxonomy_sampled_tickets.index
]
prompt_development_ticket_indices = (
    taxonomy_sampled_tickets.groupby("sampling_label", group_keys=False)
    .sample(n=1, random_state=taxonomy_random_seed)
    .index
)
validation_ticket_indices = taxonomy_sampled_tickets.index.difference(
    prompt_development_ticket_indices
)
prompt_development_tickets = prepared_tickets.loc[
    prompt_development_ticket_indices
].copy()
validation_tickets = prepared_tickets.loc[validation_ticket_indices].copy()
sampled_source_counts = (
    sampling_labels.loc[sampled_ticket_indices]
    .value_counts()
    .rename_axis("main_contact_reason")
    .reset_index(name="sampled_tickets")
)
prompt_source_counts = (
    sampling_labels.loc[prompt_development_ticket_indices]
    .value_counts()
    .rename_axis("main_contact_reason")
    .reset_index(name="prompt_development_tickets")
)
validation_source_counts = (
    sampling_labels.loc[validation_ticket_indices]
    .value_counts()
    .rename_axis("main_contact_reason")
    .reset_index(name="validation_tickets")
)
available_source_counts = (
    sampling_labels.value_counts()
    .rename_axis("main_contact_reason")
    .reset_index(name="available_tickets")
)
taxonomy_sampling_profile = (
    available_source_counts.merge(
        sampled_source_counts, on="main_contact_reason", how="left"
    )
    .merge(prompt_source_counts, on="main_contact_reason", how="left")
    .merge(validation_source_counts, on="main_contact_reason", how="left")
    .fillna(0)
    .astype(
        {
            "sampled_tickets": "int64",
            "prompt_development_tickets": "int64",
            "validation_tickets": "int64",
        }
    )
    .sort_values("main_contact_reason")
)

taxonomy_sampling_profile

#### *Findings*

The 30-ticket review sample covers all 15 observed `main_contact_reason` values twice, including the 53 tickets with no recorded source label. Each 15-ticket split therefore preserves one ticket per source-label group.

---

### Template Generation

Writes separate local CSVs with one target row per unique normalized message and blank `theme` and `sentiment` fields. Each row retains a stable message key and its duplicate-occurrence count. Sentiment is selected as `positive`, `negative`, or `neutral`. **Each file is created only once so rerunning the notebook preserves completed manual labels.**

In [ ]:
prompt_development_path = (
    project_root / "enrichment" / "data" / "prompt_development_template.csv"
)
validation_path = project_root / "enrichment" / "data" / "validation_template.csv"


def create_label_template(ticket_sample, template_path):
    existing_template = (
        pd.read_csv(template_path, sep=None, engine="python", keep_default_na=False)
        if template_path.is_file()
        else None
    )
    if (
        existing_template is not None
        and "theme" in existing_template.columns
        and existing_template["theme"].ne("").any()
    ):
        return existing_template

    template_tickets = ticket_sample[["ticket_id"]].copy()
    template_tickets["customer_messages"] = message_lists.loc[template_tickets.index]
    template_tickets["message_position"] = template_tickets["customer_messages"].map(
        lambda messages: list(range(1, len(messages) + 1))
    )
    label_template = (
        template_tickets
        .explode(["customer_messages", "message_position"], ignore_index=True)
        .rename(columns={"customer_messages": "message_text"})
        .assign(theme="", sentiment="")
    )
    label_template["normalized_message"] = label_template["message_text"].map(
        lambda message: " ".join(message.split())
    )
    label_template["message_key"] = label_template["normalized_message"].map(
        lambda message: hashlib.sha256(message.encode()).hexdigest()
    )
    label_template["message_occurrences"] = label_template.groupby(
        "message_key"
    )["message_key"].transform("size")
    label_template = label_template.drop_duplicates("message_key").drop(
        columns="normalized_message"
    )
    label_template.to_csv(template_path, index=False)
    return label_template


prompt_development_template = create_label_template(
    prompt_development_tickets, prompt_development_path
)
validation_template = create_label_template(validation_tickets, validation_path)
excluded_validation_ticket_ids = {"ticket_6MKG5_wc6_5GPaN0Kn"}
validation_template = validation_template.loc[
    ~validation_template["ticket_id"].isin(excluded_validation_ticket_ids)
].copy()
validation_template.to_csv(validation_path, index=False)

label_template_export = pd.DataFrame(
    {
        "artifact": [
            prompt_development_path.relative_to(project_root),
            validation_path.relative_to(project_root),
        ],
        "purpose": ["prompt development", "held-out validation"],
        "tickets": [
            prompt_development_template["ticket_id"].nunique(),
            validation_template["ticket_id"].nunique(),
        ],
        "unique_messages": [
            len(prompt_development_template),
            len(validation_template),
        ],
        "labelled_messages": [
            prompt_development_template["theme"].ne("").sum(),
            validation_template["theme"].ne("").sum(),
        ],
    }
)

label_template_export

#### *Findings*

The prompt-development template contains 15 tickets and the validation template contains 14. Duplicate occurrences are counted but only labelled once. The validation template excludes `ticket_6MKG5_wc6_5GPaN0Kn` because its text is predominantly quoted CookUnity staff correspondence rather than customer-authored content. Source-system contact labels are omitted, so the text-derived taxonomy can challenge rather than reproduce those existing labels.

---

### Data Labelling

Creates local, unlabelled `prompt_development_template.csv` and `validation_template.csv` files from the sampled messages. Each unique normalized message is labelled once, then its label can be mapped to duplicate occurrences.

Both templates are labelled manually, separately, and without model output. **The resulting theme and sentiment labels are committed without raw ticket text in `reference_labels.csv`.** This notebook applies that label-only artifact to freshly generated local templates, so anyone who re-ingests the source data recreates the same labelled files without committing sensitive messages. Each sentiment is selected as `positive`, `negative`, or `neutral`. The prompt-development labels define controlled themes and supply examples for the classification prompt. The held-out validation labels create the reference used to evaluate the classifier.

**A production-grade system would define its exact theme categories and positive, negative, and neutral criteria with Marketing, CX, or Product teams.** Their input would align the labels with the decisions the organisation needs to make.

In [ ]:
reference_labels_path = project_root / "enrichment" / "data" / "reference_labels.csv"
reference_labels = pd.read_csv(reference_labels_path)


def apply_reference_labels(template, template_name):
    ordered_template = template.sort_values("message_key").reset_index(drop=True).copy()
    ordered_labels = (
        reference_labels.loc[reference_labels["template"].eq(template_name)]
        .sort_values("label_index")
        .reset_index(drop=True)
    )
    if len(ordered_template) != len(ordered_labels):
        raise ValueError(
            f"{template_name} template and reference-label counts do not match."
        )
    if ordered_labels["label_index"].tolist() != list(range(1, len(ordered_labels) + 1)):
        raise ValueError(f"{template_name} reference labels are not sequential.")

    ordered_template[["theme", "sentiment"]] = ordered_labels[["theme", "sentiment"]]
    return ordered_template


prompt_development_template = apply_reference_labels(
    prompt_development_template, "prompt_development"
)
prompt_development_template.to_csv(prompt_development_path, index=False)

validation_template = apply_reference_labels(validation_template, "validation")
validation_template.to_csv(validation_path, index=False)

label_application_profile = pd.DataFrame(
    {
        "template": ["prompt_development", "validation"],
        "unique_messages": [
            len(prompt_development_template),
            len(validation_template),
        ],
        "labelled_messages": [
            prompt_development_template["theme"].notna().sum(),
            validation_template["theme"].notna().sum(),
        ],
    }
)

label_application_profile

In [ ]:
prompt_development_template = pd.read_csv(
    prompt_development_path, sep=None, engine="python", keep_default_na=False
)

prompt_development_template

In [ ]:
validation_template = pd.read_csv(
    validation_path, sep=None, engine="python", keep_default_na=False
)

validation_template

#### *Findings*

The committed label-only artifact recreates both reviewed local templates without exposing raw message text in the repository. The 23-message prompt-development template and 21-message held-out validation template are judged sufficient to demonstrate the labelling, prompt-development, and evaluation process. **A production-grade system would require substantially larger, stratified samples to represent the full range of themes and sentiments reliably.**

---

## Sampling

Sampling selects a reproducible 1,000-ticket demonstration subset for local model enrichment. The 15 prompt-development tickets and 14 held-out validation tickets are excluded to prevent reviewed examples from entering the model workload. **The remaining eligible pool therefore contains 9,971 of the 10,000 prepared tickets.** The full prepared dataset remains unchanged, while the selected local artifact provides a practical workload for validating the pipeline on personal hardware.

### Stratified Demo Sample

Excludes the 29 manual-review tickets that were assigned for prompt developement and output validation, then allocates the 1,000-ticket sample proportionally across observed `main_contact_reason` values with a fixed seed. The selected tickets are written locally for the Classification section, while the profile compares eligible-source and sample coverage.

In [ ]:
enrichment_sample_size = 1_000
enrichment_sampling_seed = 42
enrichment_sample_path = project_root / "enrichment" / "data" / "enrichment_sample.parquet"
manual_review_ticket_ids = set(prompt_development_template["ticket_id"]) | set(
    validation_template["ticket_id"]
)
enrichment_sampling_pool = prepared_tickets.loc[
    ~prepared_tickets["ticket_id"].isin(manual_review_ticket_ids)
].copy()
sampling_labels = enrichment_sampling_pool["main_contact_reason"].fillna("Missing")
source_label_counts = (
    sampling_labels.value_counts()
    .rename_axis("main_contact_reason")
    .reset_index(name="source_tickets")
)
source_label_counts["expected_sample_tickets"] = (
    source_label_counts["source_tickets"]
    .div(len(enrichment_sampling_pool))
    .mul(enrichment_sample_size)
)
source_label_counts["sample_tickets"] = source_label_counts[
    "expected_sample_tickets"
].floordiv(1).astype("int64")
remaining_sample_tickets = enrichment_sample_size - source_label_counts[
    "sample_tickets"
].sum()
allocation_order = (
    source_label_counts["expected_sample_tickets"]
    .sub(source_label_counts["sample_tickets"])
    .sort_values(ascending=False)
    .index[:remaining_sample_tickets]
)
source_label_counts.loc[allocation_order, "sample_tickets"] += 1

sampled_ticket_indices = []
for row in source_label_counts.itertuples(index=False):
    label_mask = sampling_labels.eq(row.main_contact_reason)
    sampled_ticket_indices.extend(
        enrichment_sampling_pool.loc[label_mask].sample(
            n=row.sample_tickets, random_state=enrichment_sampling_seed
        ).index
    )

enrichment_sample = enrichment_sampling_pool.loc[sampled_ticket_indices].copy()
enrichment_sample.to_parquet(enrichment_sample_path, index=False)
enrichment_sample_profile = source_label_counts.assign(
    source_share=lambda frame: frame["source_tickets"].div(len(enrichment_sampling_pool)),
    sample_share=lambda frame: frame["sample_tickets"].div(len(enrichment_sample)),
).sort_values("main_contact_reason")

enrichment_sample_profile

Imports the locally saved 1,000-ticket demonstration sample and displays it as the input to Classification.

In [ ]:
enrichment_sample = pd.read_parquet(enrichment_sample_path)

enrichment_sample

#### *Findings*

The sample contains exactly 1,000 tickets from the 9,971-ticket eligible pool, while retaining every observed `main_contact_reason` value in approximately its source proportion. Prompt-development and held-out validation tickets are excluded. **This is sufficient to demonstrate local classification and downstream analysis workflows, not to support production-grade estimates for rare themes or outcome associations.**

---

### Time Coverage

Compares source and sample ticket counts by date to confirm that the demonstration subset retains the observed time window for later trend exploration.

In [ ]:
source_date_counts = (
    enrichment_sampling_pool["ticket_date"]
    .value_counts()
    .rename_axis("ticket_date")
    .reset_index(name="source_tickets")
)
sample_date_counts = (
    enrichment_sample["ticket_date"]
    .value_counts()
    .rename_axis("ticket_date")
    .reset_index(name="sample_tickets")
)
enrichment_sample_time_profile = (
    source_date_counts.merge(sample_date_counts, on="ticket_date", how="left")
    .fillna({"sample_tickets": 0})
    .astype({"sample_tickets": "int64"})
    .assign(
        source_share=lambda frame: frame["source_tickets"].div(len(enrichment_sampling_pool)),
        sample_share=lambda frame: frame["sample_tickets"].div(len(enrichment_sample)),
    )
    .sort_values("ticket_date")
)

enrichment_sample_time_profile

#### *Findings*

The sampling profile retains every observed `ticket_date` in the prepared dataset. Daily counts are smaller than the full source, so date-level findings from this subset remain directional rather than conclusive.

---

## Model Selection

To comply with data protection requirements, **the chosen model has to be run locally and, therefore, be open-source**. Since theme identification is equally or more relevant than sentiment identification, models fine-tuned for the latter are ruled out. The **Qwen3 model family** is chosen for evaluation instead. More specifically, the experiment compares **Qwen3 4B and 8B** using the project taxonomy and labelled examples, and evaluates their performance on **theme and sentiment identification**.

Local running is done via Ollama. See the [official Ollama documentation](https://docs.ollama.com/) for installation and local-runtime guidance.

### Candidate Setup

Checks that both candidate models are available in the local Ollama runtime. Download any missing candidate with `ollama pull qwen3:4b` or `ollama pull qwen3:8b` before running the benchmark.

In [ ]:
candidate_models = ["qwen3:4b", "qwen3:8b"]
local_models = list_models()
model_candidates = pd.DataFrame(
    {
        "model": candidate_models,
        "available_locally": [model in local_models for model in candidate_models],
        "size_gb": [
            round(local_models.get(model, 0) / 1_000_000_000, 1)
            for model in candidate_models
        ],
        "pull_command": [f"ollama pull {model}" for model in candidate_models],
    }
)

model_candidates

---

### Benchmark

Measures steady-state local inference on the 21 held-out labelled messages. The prompt-development template supplies the examples, while the validation template remains unseen by the prompt. Each candidate must return schema-constrained JSON containing a concise theme and one of the three allowed sentiment values.

In [ ]:
sample_messages = [
    " ".join(message.split()).casefold()
    for messages in message_lists.loc[enrichment_sample.index]
    for message in messages
]
benchmark_workload_profile = pd.DataFrame(
    {
        "metric": [
            "held_out_messages",
            "prompt_development_examples",
            "sample_tickets",
            "sample_messages",
            "sample_unique_normalized_messages",
        ],
        "value": [
            len(validation_template),
            len(prompt_development_template),
            len(enrichment_sample),
            len(sample_messages),
            len(set(sample_messages)),
        ],
    }
)

benchmark_workload_profile

Runs every available candidate against the held-out messages and summarises runtime, JSON validity, and agreement with the manual labels. **This performs local model inference and can take several minutes per candidate.**

In [ ]:
missing_candidates = model_candidates.loc[
    ~model_candidates["available_locally"], "model"
].tolist()
if missing_candidates:
    raise RuntimeError(
        "Download missing candidates before benchmarking: "
        + ", ".join(f"ollama pull {model}" for model in missing_candidates)
    )

classification_schema = {
    "type": "object",
    "properties": {
        "theme": {"type": "string"},
        "sentiment": {
            "type": "string",
            "enum": ["positive", "negative", "neutral"],
        },
    },
    "required": ["theme", "sentiment"],
    "additionalProperties": False,
}
prompt_examples = "\n\n".join(
    f"Message: {row.message_text}\nTheme: {row.theme}\nSentiment: {row.sentiment}"
    for row in prompt_development_template.itertuples(index=False)
)

def build_benchmark_prompt(message):
    return (
        "Classify this CookUnity customer message. Return a concise theme and "
        "one sentiment. Ignore quoted staff replies, headers, and signatures. "
        "Use the examples as guidance, but create a new concise theme when needed.\n\n"
        f"Examples:\n{prompt_examples}\n\nMessage: {message}"
    )

benchmark_records = []
for model in candidate_models:
    for row in validation_template.itertuples(index=False):
        try:
            prediction, elapsed_seconds = generate_json(
                model,
                build_benchmark_prompt(row.message_text),
                classification_schema,
            )
            predicted_theme = prediction.get("theme")
            predicted_sentiment = prediction.get("sentiment")
            valid_output = (
                isinstance(predicted_theme, str)
                and bool(predicted_theme.strip())
                and predicted_sentiment in {"positive", "negative", "neutral"}
            )
            error = "" if valid_output else "Schema values were invalid."
        except RuntimeError as exception:
            elapsed_seconds = float("nan")
            predicted_theme = ""
            predicted_sentiment = ""
            valid_output = False
            error = str(exception)
        benchmark_records.append(
            {
                "model": model,
                "message_text": row.message_text,
                "expected_theme": row.theme,
                "predicted_theme": predicted_theme,
                "expected_sentiment": row.sentiment,
                "predicted_sentiment": predicted_sentiment,
                "elapsed_seconds": elapsed_seconds,
                "valid_output": valid_output,
                "theme_match": valid_output
                and predicted_theme.strip().casefold() == row.theme.strip().casefold(),
                "sentiment_match": valid_output
                and predicted_sentiment == row.sentiment,
                "error": error,
            }
        )

sample_messages_per_ticket = len(sample_messages) / len(enrichment_sample)
benchmark_predictions = pd.DataFrame(benchmark_records)
model_benchmark_results = (
    benchmark_predictions.groupby("model", as_index=False)
    .agg(
        messages=("model", "size"),
        valid_output_rate=("valid_output", "mean"),
        median_seconds_per_message=("elapsed_seconds", "median"),
        mean_seconds_per_message=("elapsed_seconds", "mean"),
        theme_exact_match_rate=("theme_match", "mean"),
        sentiment_match_rate=("sentiment_match", "mean"),
        failed_messages=("valid_output", lambda values: (~values).sum()),
        benchmark_seconds=("elapsed_seconds", "sum"),
    )
    .assign(
        baseline_seconds_per_ticket=lambda frame: (
            frame["mean_seconds_per_message"] * sample_messages_per_ticket
        )
    )
    .sort_values("mean_seconds_per_message")
)

model_benchmark_results

#### *Findings*

**Qwen3 8B is selected for Classification.** Both candidates returned valid schema-constrained JSON for all 21 held-out messages, but Qwen3 8B achieved higher sentiment agreement with the manual labels, **81.0% compared with 61.9% for Qwen3 4B**. Its **28.6% exact-theme agreement** is only marginally higher than Qwen3 4B's **23.8%**, which is expected because free-form theme wording can differ without changing meaning. Qwen3 8B takes **2.19 mean seconds per message**, compared with **1.46 seconds for Qwen3 4B**.

This performance is not sufficient for a production-grade system, but it is judged sufficient and not practically improvable for this demonstration. **The available personal hardware cannot run frontier open-weight models locally, and no secure server is available for this work.** Models such as [Qwen3-235B-A22B](https://qwenlm.github.io/blog/qwen3/) and [Llama 4 Maverick](https://ai.meta.com/blog/llama-4-multimodal-intelligence/) require substantially greater infrastructure. With access to that secure infrastructure, this same local-inference pipeline and methodology are expected to produce satisfactory production results.